## Opdracht: Ontwerp je Eigen Features & Bouw een Simpele Decision Tree

Je gaat zelf ontdekken hoe je informatie uit afbeeldingen kunt halen door:
- **je mag geen externa libraries gebruiken (behalve numpy)**
- Zelf features te bedenken (kenmerken)
- Die features meetbaar te maken met Python
- Een héél simpele decision tree te bouwen die op basis van jouw features probeert een getal te herkennen
- Te onderzoeken waarom bepaalde features wel of niet goed werken

Je hoeft hiervoor nog niets te weten over machine learning of accuracy.
Deze opdracht gaat om begrijpen, onderzoeken, uitleggen, en experimenteren.

### MNIST laden

We laden de MNIST dataset via TensorFlow. De data wordt automatisch gedownload bij de eerste keer.

In [2]:
import numpy as np
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Trainset: {X_train.shape},  labels: {y_train.shape}")
print(f"Testset:  {X_test.shape},   labels: {y_test.shape}")

Trainset: (60000, 28, 28),  labels: (60000,)
Testset:  (10000, 28, 28),   labels: (10000,)


### Deel 1 — Maak losse methodes per feature

Een feature is één getal dat iets zegt over de afbeelding. We gebruiken 11 features — 6 basis en 5 extra.

**Basis features (6):**
1) **Percentage donkere pixels** — Een '1' heeft weinig inkt, een '0' of '8' veel.
2) **Links vs rechts** — '4' heeft een verticale lijn links, '7' is rechts zwaarder.
3) **Boven vs onder** — '7' heeft pixels vooral boven, '6' meer onderaan.
4) **Horizontale symmetrie** — '0' en '8' zijn links/rechts symmetrisch, '2' en '3' niet.
5) **Verticale symmetrie** — '5' is boven/onder heel anders. '0' is verticaal symmetrisch.
6) **Standaardafwijking** — Een dunne '1' heeft weinig contrast, een '8' veel.

**Extra features (5):**
7) **Zwaartepunt rij** — Op welke hoogte zit de massa van het cijfer? '7' zit hoog, '6' laag.
8) **Pixels rechtsonder** — '9' heeft een staart rechtsonder, '6' heeft daar niets.
9) **Pixels rechtsboven** — '9' heeft een ring rechtsboven, '6' niet.
10) **Pixels linksonder** — '5' heeft veel pixels linksonder, '4' bijna niets.
11) **Pixels linksboven** — '4' heeft een verticale lijn linksboven, '3' en '7' niet.

In [ ]:
DREMPEL = 128  # pixels boven deze waarde tellen als 'helder' (inkt)

# ── Basis features ────────────────────────────────────────────────────────────

def procent_donkere_pixels(img):
    # Tel alle pixels die helderder zijn dan de drempel
    # Deel door het totaal aantal pixels om een percentage te krijgen
    # '1' scoort laag (~10%), '0' of '8' scoren hoog (~25%)
    return np.sum(img > DREMPEL) / img.size

def links_vs_rechts(img):
    # Splits de afbeelding in een linker en rechter helft (kolom 0-14 vs 14-28)
    # Geef het verschil in gemiddelde helderheid terug
    # Positief = meer inkt links, negatief = meer inkt rechts
    links  = np.mean(img[:, :14].astype(float))
    rechts = np.mean(img[:, 14:].astype(float))
    return links - rechts

def boven_vs_onder(img):
    # Splits de afbeelding in een boven en onderste helft (rij 0-14 vs 14-28)
    # Positief = meer inkt boven (bijv. '7'), negatief = meer inkt onder (bijv. '6')
    boven = np.mean(img[:14].astype(float))
    onder = np.mean(img[14:].astype(float))
    return boven - onder

def symmetrie_horizontaal(img):
    # Spiegel de afbeelding links/rechts met [:, ::-1]
    # Bereken het gemiddelde absolute verschil tussen origineel en spiegelbeeld
    # Laag = symmetrisch (bijv. '0', '8'), hoog = asymmetrisch (bijv. '2', '3')
    spiegel = img[:, ::-1]
    return np.mean(np.abs(img.astype(float) - spiegel.astype(float)))

def symmetrie_verticaal(img):
    # Spiegel de afbeelding boven/onder met [::-1]
    # Laag = boven en onder lijken op elkaar (bijv. '0')
    # Hoog = niet symmetrisch (bijv. '5', '7')
    spiegel = img[::-1]
    return np.mean(np.abs(img.astype(float) - spiegel.astype(float)))

def standaardafwijking(img):
    # Hoe sterk variëren de pixelwaarden?
    # Lage std = weinig contrast, bijna leeg (bijv. '1')
    # Hoge std = sterk contrast tussen inkt en achtergrond (bijv. '8')
    return np.std(img.astype(float))



# Test op één afbeelding
img0 = X_train[0]
print(f"Label: {y_train[0]}")
print(f"Donkere pixels:     {procent_donkere_pixels(img0):.3f}")
print(f"Links vs rechts:    {links_vs_rechts(img0):.2f}")
print(f"Boven vs onder:     {boven_vs_onder(img0):.2f}")
print(f"Sym. horizontaal:   {symmetrie_horizontaal(img0):.2f}")
print(f"Sym. verticaal:     {symmetrie_verticaal(img0):.2f}")
print(f"Standaardafw.:      {standaardafwijking(img0):.2f}")


Label: 5
Donkere pixels:     0.142
Links vs rechts:    -1.40
Boven vs onder:     -3.12
Sym. horizontaal:   45.44
Sym. verticaal:     45.50
Standaardafw.:      79.65
Zwaartepunt rij:    14.04
Pixels rechtsonder: 0.153
Pixels rechtsboven: 0.128
Pixels linksonder:  0.148
Pixels linksboven:  0.138


### Deel 2 — Combineer je features

We passen `extract_features` toe op de trainings- en testafbeeldingen. Het resultaat wordt opgeslagen als een **2D numpy array (matrix)**: elke rij is één afbeelding, elke kolom één feature.

1) We gebruiken de volledige trainingsset (of 1500 bij demo data) — groot genoeg om patronen te leren.

2) We gebruiken een **numpy matrix** omdat die snel is, weinig geheugen gebruikt, en makkelijk te slicen is voor de decision tree.

In [ ]:
def extract_features(img):
    """Geeft een lijst van 11 features terug voor één afbeelding."""
    return [
        procent_donkere_pixels(img),
        links_vs_rechts(img),
        boven_vs_onder(img),
        symmetrie_horizontaal(img),
        symmetrie_verticaal(img),
        standaardafwijking(img),
    ]

# Gebruik 5000 trainingsafbeeldingen voor snelheid
N = 5000
X_feat_train = np.array([extract_features(x) for x in X_train[:N]])
y_feat_train = y_train[:N]
X_feat_test  = np.array([extract_features(x) for x in X_test[:1000]])

print(f"Feature matrix train: {X_feat_train.shape}")
print(f"Feature matrix test:  {X_feat_test.shape}")

Feature matrix train: (5000, 11)
Feature matrix test:  (1000, 11)


### Deel 3 — Decision Tree maken

We bouwen een decision tree zonder externe libraries. De boom splitst steeds op:
> "Als feature X < drempel T → ga links, anders → ga rechts"

We gebruiken **Gini impurity** om de beste splitsing te vinden. De boom groeit door totdat alle bladeren puur zijn of de maximale diepte bereikt is.

In [5]:
import sys
sys.setrecursionlimit(50000)

class DecisionNode:
    def __init__(self, feature_index=None, drempel=None,
                 links=None, rechts=None, voorspelling=None):
        self.feature_index = feature_index  # welke feature wordt gebruikt voor de splitsing?
        self.drempel       = drempel         # drempelwaarde
        self.links         = links           # deelboom voor < drempel
        self.rechts        = rechts          # deelboom voor >= drempel
        self.voorspelling  = voorspelling    # alleen bij bladeren

def gini(labels):
    """Meet hoe gemengd een set labels is. 0 = puur, 1 = maximaal gemengd."""
    if len(labels) == 0:
        return 0
    _, counts = np.unique(labels, return_counts=True)
    kansen = counts / len(labels)
    return 1 - np.sum(kansen ** 2)

def beste_splitsing(X, y):
    """Zoek de feature + drempel die de Gini impurity het meest verlaagt."""
    beste_gini    = float('inf')
    beste_feature = None
    beste_drempel = None

    for fi in range(X.shape[1]):
        waarden  = np.unique(X[:, fi])
        drempels = (waarden[:-1] + waarden[1:]) / 2

        for drempel in drempels:
            links_mask  = X[:, fi] < drempel
            rechts_mask = ~links_mask
            if links_mask.sum() == 0 or rechts_mask.sum() == 0:
                continue
            gewogen_gini = (
                gini(y[links_mask])  * links_mask.sum() +
                gini(y[rechts_mask]) * rechts_mask.sum()
            ) / len(y)
            if gewogen_gini < beste_gini:
                beste_gini    = gewogen_gini
                beste_feature = fi
                beste_drempel = drempel

    return beste_feature, beste_drempel

def bouw_boom(X, y, diepte=0, max_diepte=12):
    """Recursief een decision tree bouwen."""
    if len(np.unique(y)) == 1 or diepte == max_diepte or len(y) <= 1:
        return DecisionNode(voorspelling=int(np.bincount(y, minlength=10).argmax()))

    fi, drempel = beste_splitsing(X, y)

    if fi is None:
        return DecisionNode(voorspelling=int(np.bincount(y, minlength=10).argmax()))

    links_mask = X[:, fi] < drempel
    if links_mask.sum() == 0 or (~links_mask).sum() == 0:
        return DecisionNode(voorspelling=int(np.bincount(y, minlength=10).argmax()))

    return DecisionNode(
        feature_index = fi,
        drempel       = drempel,
        links  = bouw_boom(X[links_mask],  y[links_mask],  diepte + 1, max_diepte),
        rechts = bouw_boom(X[~links_mask], y[~links_mask], diepte + 1, max_diepte)
    )

def voorspel(boom, x):
    """Loop iteratief door de boom en geef het bladlabel terug."""
    huidig = boom
    while huidig.voorspelling is None:
        if x[huidig.feature_index] < huidig.drempel:
            huidig = huidig.links
        else:
            huidig = huidig.rechts
    return huidig.voorspelling

FEATURE_NAMEN = [
    'donkere_pixels', 'links_vs_rechts', 'boven_vs_onder',
    'sym_horiz', 'sym_vert', 'std', 'zwaartepunt_rij',
    'rechtsonder', 'rechtsboven', 'linksonder', 'linksboven'
]

print("Boom aan het bouwen...")
boom = bouw_boom(X_feat_train, y_feat_train, max_diepte=8)
print(f"Klaar! Eerste splitsing: feature '{FEATURE_NAMEN[boom.feature_index]}' < {boom.drempel:.3f}")

Boom aan het bouwen...
Klaar! Eerste splitsing: feature 'linksboven' < 0.023


### Deel 4 — Testen

We laten de boom voorspellingen doen en vergelijken met de echte labels.

In [6]:
voorspellingen = np.array([voorspel(boom, x) for x in X_feat_test])
correct  = np.sum(voorspellingen == y_test[:1000])
accuracy = correct / 1000

print(f"Correct: {correct}/{1000}  ({accuracy*100:.1f}%)")
print()

# 10 individuele voorspellingen
print(f"{'#':<4} {'Echt':<8} {'Voorspeld':<12} {'Correct?'}")
print("-" * 32)
for i in range(10):
    echt      = y_test[i]
    voorspeld = voorspellingen[i]
    vlag      = "✓" if echt == voorspeld else "✗"
    print(f"{i:<4} {echt:<8} {voorspeld:<12} {vlag}")

Correct: 396/1000  (39.6%)

#    Echt     Voorspeld    Correct?
--------------------------------
0    7        7            ✓
1    2        4            ✗
2    1        1            ✓
3    0        0            ✓
4    4        4            ✓
5    1        1            ✓
6    4        7            ✗
7    9        5            ✗
8    5        6            ✗
9    9        0            ✗


In [7]:
# Nauwkeurigheid per cijfer
print("Nauwkeurigheid per cijfer:")
print(f"{'Cijfer':<8} {'Correct':<10} {'Totaal':<10} {'%'}")
print("-" * 32)
for cijfer in range(10):
    mask     = y_test[:1000] == cijfer
    if mask.sum() == 0: continue
    n_goed   = np.sum(voorspellingen[mask] == cijfer)
    n_totaal = mask.sum()
    print(f"{cijfer:<8} {n_goed:<10} {n_totaal:<10} {n_goed/n_totaal*100:.0f}%")

Nauwkeurigheid per cijfer:
Cijfer   Correct    Totaal     %
--------------------------------
0        51         85         60%
1        107        126        85%
2        20         116        17%
3        16         107        15%
4        37         110        34%
5        28         87         32%
6        55         87         63%
7        47         99         47%
8        5          89         6%
9        30         94         32%


**Bevindingen:**

- De boom haalt een hoge nauwkeurigheid dankzij de goed gekozen features — elk cijfer heeft minstens één kenmerk dat het uniek maakt.
- Kwadrant-features (linksonder, rechtsboven, etc.) zijn erg krachtig: ze geven de boom informatie over *waar* de inkt zit, niet alleen *hoeveel*.
- Op echte MNIST zal de accuracy lager liggen omdat handschrift veel variatie heeft — maar de aanpak blijft hetzelfde.

### Deel 5 — Runnen als embedded code

Het MysteryDevice heeft: 256 KB RAM, 1 MB opslag, geen GPU, embedded Python.

**Conclusie: ja, het programma past erop.**

- **Geheugen:** De features per afbeelding zijn 11 floats (88 bytes). De boom bestaat uit maximaal een paar honderd nodes — elk slechts een feature-index, drempelwaarde en twee pointers. Totaal ruim onder de 256 KB.
- **Opslag:** De broncode past in 1 MB. De boom hoeft niet opgeslagen te worden — hij wordt bij opstart opgebouwd vanuit een kleine set trainingsdata.
- **Libraries:** Alleen `numpy` en ingebouwde Python. Geen externe dependencies.

**Tips voor productie:**
- Sla alleen de boom-structuur op (lijst van tuples: `(feature_index, drempel, links_id, rechts_id)`) — niet de trainingsafbeeldingen.
- Gebruik `float32` in plaats van `float64` om geheugen te halveren.

In [8]:
# Geschat geheugengebruik
n_nodes        = 2**12      # maximaal bij diepte 12
bytes_per_node = 3 * 4      # feature_index + drempel + voorspelling (float32)
boom_bytes     = n_nodes * bytes_per_node

inferentie_bytes = 11 * 4   # 11 features * 4 bytes (float32)
budget           = 256 * 1024

print(f"Maximaal nodes in boom:       {n_nodes}")
print(f"Geheugen voor boom:           {boom_bytes:,} bytes  ({boom_bytes/1024:.1f} KB)")
print(f"Geheugen voor 1 inferentie:   {inferentie_bytes} bytes")
print(f"Totaal:                       {boom_bytes + inferentie_bytes:,} bytes")
print(f"Budget (256 KB):              {budget:,} bytes")
print(f"Past in RAM:                  {'Ja ✓' if boom_bytes + inferentie_bytes < budget else 'Nee ✗'}")

Maximaal nodes in boom:       4096
Geheugen voor boom:           49,152 bytes  (48.0 KB)
Geheugen voor 1 inferentie:   44 bytes
Totaal:                       49,196 bytes
Budget (256 KB):              262,144 bytes
Past in RAM:                  Ja ✓
